# Energy demand prediction (reproducible ML workflow)

This notebook is a reproducible companion to the blog post `Machine_Learning_Based_Energy_Demand_Prediction_Blog_Post.md`.

## Goals

- Build a leakage-aware dataset for **hourly energy demand forecasting**
- Compare a **naive baseline** vs **ML models**
- Report **MAE/RMSE** and visualize predictions

> Note: The dataset used below is **synthetic but realistic** so the notebook runs anywhere. Replace the data section with your real grid/utility dataset to replicate the paper more directly.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

plt.rcParams["figure.figsize"] = (12, 4)


In [ ]:
def make_demo_data(n_hours: int = 24 * 365, seed: int = 7) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    idx = pd.date_range("2022-01-01", periods=n_hours, freq="h")
    df = pd.DataFrame(index=idx)

    df["hour"] = df.index.hour
    df["dow"] = df.index.dayofweek
    df["month"] = df.index.month

    temp_daily = 8 * np.sin(2 * np.pi * (df["hour"] / 24.0) - 1.2)
    temp_season = 12 * np.sin(2 * np.pi * (df.index.dayofyear / 365.25) - 0.5)
    df["temperature_c"] = 18 + temp_daily + temp_season + rng.normal(0, 1.0, size=len(df))

    is_weekend = (df["dow"] >= 5).astype(int)
    base = 1200 + 80 * np.cos(2 * np.pi * (df["hour"] / 24.0))
    heating = np.clip(18 - df["temperature_c"], 0, None) * 45
    cooling = np.clip(df["temperature_c"] - 22, 0, None) * 55
    weekend_drop = is_weekend * 120
    noise = rng.normal(0, 35, size=len(df))

    df["demand_mw"] = base + heating + cooling - weekend_drop + noise

    # Leakage-safe lags (shift) and rolling stats (computed on shifted series)
    df["lag_1"] = df["demand_mw"].shift(1)
    df["lag_24"] = df["demand_mw"].shift(24)
    df["lag_168"] = df["demand_mw"].shift(168)
    df["roll_mean_24"] = df["demand_mw"].shift(1).rolling(24).mean()
    df["roll_std_24"] = df["demand_mw"].shift(1).rolling(24).std()

    df = df.dropna().reset_index(names="timestamp")
    return df


df = make_demo_data()
df.head()

In [ ]:
# Quick EDA
ax = df.set_index("timestamp")["demand_mw"].iloc[: 14 * 24].plot()
ax.set_title("Synthetic energy demand (first 2 weeks)")
ax.set_ylabel("Demand (MW)")
plt.show()

df[["demand_mw", "temperature_c"]].describe().T

In [ ]:
def time_split(df: pd.DataFrame, train_frac: float = 0.8):
    n = len(df)
    split = int(n * train_frac)
    return df.iloc[:split].copy(), df.iloc[split:].copy()

train, test = time_split(df, 0.8)
print(len(train), len(test), train["timestamp"].min(), train["timestamp"].max())
print(test["timestamp"].min(), test["timestamp"].max())


In [ ]:
target = "demand_mw"
cat_cols = ["dow", "month"]
num_cols = [
    "hour",
    "temperature_c",
    "lag_1",
    "lag_24",
    "lag_168",
    "roll_mean_24",
    "roll_std_24",
]

X_train = train[num_cols + cat_cols]
y_train = train[target].to_numpy()
X_test = test[num_cols + cat_cols]
y_test = test[target].to_numpy()

# Baseline: persistence using lag_1 (predict next hour = last hour)
baseline_pred = X_test["lag_1"].to_numpy()

def metrics(y, yhat):
    mae = mean_absolute_error(y, yhat)
    rmse = np.sqrt(mean_squared_error(y, yhat))
    return {"mae": mae, "rmse": rmse}

baseline_metrics = metrics(y_test, baseline_pred)
baseline_metrics

In [ ]:
pre = ColumnTransformer(
    transformers=[
        ("num", "passthrough", num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ]
)

models = {
    "Ridge": Ridge(alpha=1.0, random_state=42),
    "HistGradientBoosting": HistGradientBoostingRegressor(
        random_state=42,
        max_depth=6,
        learning_rate=0.08,
    ),
}

results = []

for name, reg in models.items():
    pipe = Pipeline([( "pre", pre), ("reg", reg)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    m = metrics(y_test, pred)
    results.append({"model": name, **m})

results.append({"model": "Baseline (lag_1)", **baseline_metrics})

pd.DataFrame(results).sort_values("mae")

In [ ]:
# Plot actual vs predicted for the best model on the first test week
best_name = pd.DataFrame(results).sort_values("mae").iloc[0]["model"]
print("Best model:", best_name)

best_reg = models[best_name] if best_name in models else None
pipe = Pipeline([( "pre", pre), ("reg", best_reg)])
pipe.fit(X_train, y_train)
pred = pipe.predict(X_test)

plot_df = test[["timestamp", "demand_mw"]].copy()
plot_df["predicted"] = pred
plot_df = plot_df.iloc[: 7 * 24]

ax = plot_df.set_index("timestamp")[["demand_mw", "predicted"]].plot()
ax.set_title("Actual vs predicted demand (first test week)")
ax.set_ylabel("Demand (MW)")
plt.show()
